**Cell 1**

# NB13.1 — Method comparison with three reward models

This notebook repeats the NB13 coefficient-method comparison for the five exact
mean DPO experts from NB11.1 and evaluates every frozen generated answer with:

1. `RLHFlow/ArmoRM-Llama3-8B-v0.1` (primary evaluator),
2. `LxzGordon/URM-LLaMa-3.1-8B` (sensitivity evaluator; trained on HelpSteer2),
3. `nvidia/Llama2-13B-SteerLM-RM` (external robustness evaluator).

Every merge point is generated **once**. The prompt/answer pairs are cached on
Google Drive and then scored by all three RMs, so evaluator comparisons use
identical texts. Raw scores are never averaged across RMs because their scales
and training data differ.

Cert is first audited against the independent floor LP. If every Phase-B
preference has a collapsed floor and `lambda_cert = p` within tolerance, Cert is
kept only in `cert_floor_audit.csv` and omitted from generation, RM evaluation,
method tables, and hypothesis tests. If even one Phase-B preference is
non-trivial, Cert remains in the complete comparison.

ArmoRM and URM run directly in Colab, one after the other. The official
SteerLM-RM is a 26 GB NeMo checkpoint and must be served through NVIDIA's
NeMo-Aligner PyTriton server. Section 10 records the exact server command and
requires an explicit checkpoint attestation. All long-running stages are
resumable from Drive.


**Cell 2**

## 1. Mount Drive, clone the repository, and load the NB11.1 mean adapters


In [ ]:
# Cell 3
# Drive is intentionally the first interactive action in "Run all".
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

%cd /content
import hashlib, json, os, shutil, stat, zipfile
from pathlib import Path

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

ADAPTER_AXES = ("helpfulness", "correctness", "coherence", "complexity", "verbosity")
NB11_1_RUN_TAG = "nb11_1_multiseed_pair137_train137_141"
BUNDLE_NAME = "nb11_1_tinyllama_helpsteer2_dpo_multiseed_mean_adapters.zip"
drive_bundle = (
    Path("/content/drive/MyDrive/master-thesis-nb11-1")
    / NB11_1_RUN_TAG / BUNDLE_NAME
)
candidates = [path for path in (
    drive_bundle,
    Path("/content") / BUNDLE_NAME,
    Path.cwd() / BUNDLE_NAME,
) if path.is_file()]
if not candidates:
    from google.colab import files
    print(f"Bitte jetzt {BUNDLE_NAME} auswählen.")
    uploaded = files.upload()
    candidates = [Path.cwd() / name for name in uploaded if name == BUNDLE_NAME]
if not candidates:
    raise FileNotFoundError(f"Required NB11.1 bundle not found: {BUNDLE_NAME}")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Duplicate copies are harmless only when their bytes agree.
candidate_hashes = {str(path.resolve()): sha256_file(path) for path in candidates}
if len(set(candidate_hashes.values())) != 1:
    raise RuntimeError(f"Conflicting NB11.1 bundles found: {candidate_hashes}")
ADAPTER_BUNDLE_PATH = candidates[0].resolve()
ADAPTER_BUNDLE_SHA256 = next(iter(candidate_hashes.values()))
ADAPTER_INPUT_ROOT = Path("/content/nb13_1_inputs") / ADAPTER_BUNDLE_SHA256[:16] / "mean_adapters"
RUN_MANIFEST_PATH = ADAPTER_INPUT_ROOT.parent / "geometry/nb11_1_run_manifest.json"

required_members = {
    "geometry/nb11_1_run_manifest.json",
    *{
        f"mean_adapters/dpo_{axis}/adapter/{filename}"
        for axis in ADAPTER_AXES
        for filename in (
            "adapter_config.json", "adapter_model.safetensors", "averaging_manifest.json"
        )
    },
}
required_on_disk = [ADAPTER_INPUT_ROOT.parent / member for member in sorted(required_members)]
if not all(path.is_file() for path in required_on_disk):
    with zipfile.ZipFile(ADAPTER_BUNDLE_PATH) as bundle:
        entries = {entry.filename: entry for entry in bundle.infolist()}
        missing = sorted(required_members - set(entries))
        if missing:
            raise FileNotFoundError(f"NB11.1 bundle is incomplete; missing={missing}")
        payload_size = sum(entries[name].file_size for name in required_members)
        if payload_size > 1_500_000_000:
            raise RuntimeError(f"Unexpectedly large adapter payload: {payload_size} bytes")
        target = ADAPTER_INPUT_ROOT.parent.resolve()
        target.mkdir(parents=True, exist_ok=True)
        for name in sorted(required_members):
            entry = entries[name]
            unix_mode = (entry.external_attr >> 16) & 0o170000
            if unix_mode == stat.S_IFLNK:
                raise RuntimeError(f"Symlink forbidden in adapter ZIP: {name}")
            destination = (target / name).resolve()
            if target != destination and target not in destination.parents:
                raise RuntimeError(f"Unsafe path in adapter ZIP: {name}")
            destination.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(entry) as source, destination.open("wb") as output:
                shutil.copyfileobj(source, output)

missing = [str(path) for path in required_on_disk if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Extraction incomplete: {missing}")

run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding="utf-8"))
assert run_manifest["run_tag"] == NB11_1_RUN_TAG
assert run_manifest["training_seeds"] == [137, 138, 139, 140, 141]
assert run_manifest["pair_seed"] == 137
assert run_manifest["attribute_order"] == list(ADAPTER_AXES)
assert run_manifest["mean_operation"] == "exact arithmetic mean of effective LoRA updates"
assert run_manifest["mean_output_rank"] == 40

MEAN_MANIFESTS = {}
ADAPTER_PATHS = {}
for axis in ADAPTER_AXES:
    adapter = ADAPTER_INPUT_ROOT / f"dpo_{axis}/adapter"
    manifest_path = adapter / "averaging_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    config = json.loads((adapter / "adapter_config.json").read_text(encoding="utf-8"))
    assert manifest["operation"] == "exact_arithmetic_mean_of_effective_lora_updates"
    assert manifest["label"] == axis
    assert manifest["source_count"] == 5
    assert [entry["seed"] for entry in manifest["source_adapters"]] == [137, 138, 139, 140, 141]
    assert manifest["source_rank"] == 8 and manifest["output_rank"] == 40
    assert manifest["output_lora_alpha"] == 16
    assert manifest["verification_relative_frobenius_error"] <= 1e-6
    assert config["r"] == 40 and config["lora_alpha"] == 16
    assert manifest["adapter_model_sha256"] == sha256_file(adapter / "adapter_model.safetensors")
    assert manifest["adapter_config_sha256"] == sha256_file(adapter / "adapter_config.json")
    assert run_manifest["mean_adapter_hashes"][axis] == manifest["adapter_model_sha256"]
    MEAN_MANIFESTS[axis] = manifest
    ADAPTER_PATHS[axis] = adapter.resolve()
    print(f"[OK] mean {axis:12s} {manifest['adapter_model_sha256'][:16]}...")
print(f"[OK] NB11.1 bundle SHA256 = {ADAPTER_BUNDLE_SHA256}")


**Cell 4**

## 2. Runtime and dependencies


In [ ]:
# Cell 5
!nvidia-smi || echo "No local GPU. Generation, ArmoRM and URM require one."


In [ ]:
# Cell 6
import importlib.metadata, subprocess, sys

print(f"Python {sys.version.split()[0]} ({sys.executable})")
packages = [
    "pandas==2.2.3", "numpy==2.1.3", "scipy==1.14.1",
    "protobuf==5.29.5", "transformers==4.45.2", "tokenizers==0.20.3",
    "peft==0.13.2", "accelerate==1.1.1", "datasets==3.1.0",
    "huggingface_hub==0.36.0", "bitsandbytes==0.50.2", "pyyaml", "safetensors", "psutil",
    "nvidia-pytriton==0.7.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "--only-binary=:all:", *packages],
    check=True,
)
expected = {"transformers": "4.45.2", "tokenizers": "0.20.3", "peft": "0.13.2"}
installed = {name: importlib.metadata.version(name) for name in expected}
assert installed == expected, f"Unexpected inference stack: {installed}"
print("Runtime:", {name: importlib.metadata.version(name) for name in
       ("transformers", "tokenizers", "peft", "accelerate", "datasets", "nvidia-pytriton")})


**Cell 7**

## 3. Frozen settings and output paths


In [ ]:
# Cell 8
from __future__ import annotations

import gc, math, re, time
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/content/master-thesis").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_TAG = "nb13_1_nb11_1_mean_dpo_three_rms_run1"
CLAIM_STATUS = "exploratory"
CONFIG_PATH = PROJECT_ROOT / "configs/tinyllama_helpsteer2_armorm.yaml"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/13.1_method_comparison_helpsteer2_dpo_three_rms_colab.ipynb"
RESULTS_DIR = PROJECT_ROOT / "results/nb13_1_helpsteer2_dpo_three_rm_method_comparison" / RUN_TAG
PERSISTENT_DIR = Path("/content/drive/MyDrive/master-thesis-nb13-1") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)

LAMBDA_ALL_CSV = RESULTS_DIR / "lambda_table_with_cert_audit.csv"
LAMBDA_CSV = RESULTS_DIR / "lambda_table_evaluated.csv"
CERT_AUDIT_CSV = RESULTS_DIR / "cert_floor_audit.csv"
CERT_AUDIT_JSON = RESULTS_DIR / "cert_floor_audit.json"
PREREG_JSON = PERSISTENT_DIR / "nb13_1_preregistration.json"
ANSWER_CACHE = PERSISTENT_DIR / "answer_cache.jsonl"
REWARD_PROMPT_PATH = RESULTS_DIR / "nb13_1_reward_prompts.jsonl"
DIAGNOSTIC_EXCLUSION_PATH = RESULTS_DIR / "nb11_diagnostic_prompts_exclusion.jsonl"
COSINE_MATRIX_CSV = RESULTS_DIR / "R_mean_dpo_cos.csv"
GRAM_MATRIX_CSV = RESULTS_DIR / "R_mean_dpo_gram.csv"
D_NORMS_CSV = RESULTS_DIR / "mean_dpo_update_norms.csv"
GEOMETRY_JSON = RESULTS_DIR / "mean_dpo_geometry_report.json"
FINAL_CSV = RESULTS_DIR / "method_comparison_three_rms.csv"
STATS_CSV = RESULTS_DIR / "paired_statistics_three_rms.csv"
ROBUST_CSV = RESULTS_DIR / "normalization_robustness_three_rms.csv"
PROXY_CSV = RESULTS_DIR / "proxy_method_diagnostics_three_rms.csv"
PROXY_JSON = RESULTS_DIR / "proxy_validation_three_rms.json"
RM_AGREEMENT_CSV = RESULTS_DIR / "reward_model_agreement.csv"
REPORT_JSON = RESULTS_DIR / "nb13_1_report.json"

PRIMARY_MATRIX = "R_cos"
RHO_GRID = [0.0, 0.1, 0.2, 0.5]
CERT_C = 0.5
CERT_EPS = 1e-8
C_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0]
ALPHA_GRID = [0.0, 0.5, 1.0, 2.0]
EPS_GRID = [0.01, 0.02, 0.05, 0.10]
DIRICHLET_SEED = 137
FLOOR_TOL = 2e-6

PREREG_CONFIRM = True
RUN_REWARD_COLLECTION = True
N_EVAL_PROMPTS = 80
EVAL_PROMPT_SEED = 1313
MAX_NEW_TOKENS = 256
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 5
LAMBDA_DEDUP_DECIMALS = 8

ARMORM_MODEL = "RLHFlow/ArmoRM-Llama3-8B-v0.1"
ARMORM_REVISION = "eb2676d20da2f2d41082289d23c59b9f7427f955"
URM_MODEL = "LxzGordon/URM-LLaMa-3.1-8B"
URM_REVISION = "7b5a6446f55725ae6d86002fdc30f702a2272e0f"
STEERLM_MODEL = "nvidia/Llama2-13B-SteerLM-RM"
STEERLM_REVISION = "a8fd6a92a4967f934612f519a2ef078bb87ec431"
STEERLM_CHECKPOINT_SHA256 = "19851451def71de48d044bbebf93481308f1a778153e96e8b93f490047082b27"

NB11_REPORTED_DIAGNOSTIC_PROMPT_SHA256 = "3864d363dbff91fdeda62029a16b0da3b3436fe3557d42cb75739c75649ff51e"
NB11_RECONSTRUCTED_DIAGNOSTIC_PROMPT_SHA256 = "9d2c642c0413a4e3cdb8309a830aaff46c3ba5559205952fc1641e609f161a26"

print(f"run tag       = {RUN_TAG}")
print(f"results       = {RESULTS_DIR}")
print(f"persistent    = {PERSISTENT_DIR}")
print(f"claim status  = {CLAIM_STATUS}")


**Cell 9**

## 4. Compute the relationship matrix from the five mean DPO adapters


In [ ]:
# Cell 10
from src.experiment_config import get_attribute_order, load_experiment_config, validate_preference_vectors
from src.proxy_validation import (
    build_search_set, coefficient_key, normalization_agreement,
    run_spearman_analysis, safe_spearman, write_json,
)
from src.coefficient_portfolio import avg, cert, fair_alpha_eps, floor_lp_at_p, improvements, maxmin_c
from src.lambda_utils import lambda_key
from src.cert_floor_policy import should_exclude_cert_from_phase_b

config = load_experiment_config(CONFIG_PATH)
ATTRIBUTES = get_attribute_order(config)
PREFERENCES = validate_preference_vectors(config)
m = len(ATTRIBUTES)
assert tuple(ATTRIBUTES) == ADAPTER_AXES
print(f"Attribute order: {list(ATTRIBUTES)}")


In [ ]:
# Cell 11
import torch
from src.effective_lora_geometry import (
    effective_lora_inner_product, effective_lora_update_norm,
    effective_lora_update_numel, load_effective_lora_geometry,
    validate_compatible_geometries,
)
from src.preferences import PREFERENCES as PREFERENCES_MODULE

REGIME = "helpsteer2_dpo_nb11_1_exact_mean_adapters"
BASE_MODEL_NAME = run_manifest["base_model"]
BASE_REVISION = run_manifest["base_revision"]
DATASET_NAME = run_manifest["dataset"]
DATASET_REVISION = run_manifest["dataset_revision"]
assert BASE_MODEL_NAME == "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
assert BASE_REVISION == "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
assert DATASET_NAME == "nvidia/HelpSteer2"
assert DATASET_REVISION == "990b2711a36180dd19d9c94b8627844866f8982a"
assert set(PREFERENCES) == set(PREFERENCES_MODULE)
for name in PREFERENCES:
    assert np.allclose(PREFERENCES[name], PREFERENCES_MODULE[name], atol=1e-12)

geometries = {axis: load_effective_lora_geometry(ADAPTER_PATHS[axis]) for axis in ATTRIBUTES}
validate_compatible_geometries([geometries[a] for a in ATTRIBUTES], list(ATTRIBUTES))
R_gram = np.empty((m, m), dtype=np.float64)
for i, left in enumerate(ATTRIBUTES):
    for j, right in enumerate(ATTRIBUTES[:i + 1]):
        value = effective_lora_inner_product(geometries[left], geometries[right])
        R_gram[i, j] = R_gram[j, i] = value
d_norms = np.sqrt(np.maximum(np.diag(R_gram), 0.0))
for i, axis in enumerate(ATTRIBUTES):
    assert np.isclose(d_norms[i], effective_lora_update_norm(geometries[axis]), rtol=1e-10)
R_cos = R_gram / np.outer(d_norms, d_norms)
R_cos = 0.5 * (R_cos + R_cos.T)
assert np.allclose(np.diag(R_cos), 1.0, atol=1e-10)
eigenvalues = np.linalg.eigvalsh(R_cos)
assert eigenvalues.min() > 1e-10 * eigenvalues.max(), eigenvalues
R = R_cos if PRIMARY_MATRIX == "R_cos" else R_gram

pd.DataFrame(R_cos, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(COSINE_MATRIX_CSV)
pd.DataFrame(R_gram, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(GRAM_MATRIX_CSV)
pd.DataFrame({"attribute": ATTRIBUTES, "effective_update_norm": d_norms}).to_csv(D_NORMS_CSV, index=False)
offdiag = R_cos[np.triu_indices(m, 1)]
GEOMETRY_SUMMARY = {
    "regime": REGIME,
    "primary_matrix": PRIMARY_MATRIX,
    "attribute_order": list(ATTRIBUTES),
    "effective_update_numel": int(effective_lora_update_numel(geometries[ATTRIBUTES[0]])),
    "update_norms": dict(zip(ATTRIBUTES, map(float, d_norms))),
    "cosine_eigenvalues": eigenvalues.tolist(),
    "cosine_condition_number": float(eigenvalues.max() / eigenvalues.min()),
    "cosine_offdiag_min": float(offdiag.min()),
    "cosine_offdiag_max": float(offdiag.max()),
    "cosine_offdiag_mean": float(offdiag.mean()),
    "inverse_R_ones": np.linalg.solve(R, np.ones(m)).tolist(),
    "matrix_sha256": {"R_cos": sha256_file(COSINE_MATRIX_CSV), "R_gram": sha256_file(GRAM_MATRIX_CSV)},
}
write_json(GEOMETRY_JSON, GEOMETRY_SUMMARY)
display(pd.DataFrame(R_cos, index=ATTRIBUTES, columns=ATTRIBUTES).round(5))
print(json.dumps(GEOMETRY_SUMMARY, indent=2))


**Cell 12**

## 5. Phase A: frozen coefficient grid and Cert floor audit


In [ ]:
# Cell 13
PREF_SET = [(name, np.asarray(vec, dtype=np.float64)) for name, vec in PREFERENCES.items()]
seen = {tuple(np.round(v, 9)) for _, v in PREF_SET}
draws = build_search_set(
    m, n_dirichlet=64, dirichlet_alpha=1.0,
    preferences=list(PREFERENCES.values()), seed=DIRICHLET_SEED,
)
for index, vector in enumerate(draws):
    key = tuple(np.round(vector, 9))
    if key not in seen:
        seen.add(key)
        PREF_SET.append((f"dirichlet_{index:02d}", np.asarray(vector, dtype=np.float64)))
for name, p in PREF_SET:
    assert abs(float(p.sum()) - 1.0) < 1e-9 and np.all(p >= -1e-12), name

PREF_SET_B = [(name, p) for name, p in PREF_SET if name in PREFERENCES]
PHASE_B_NAMES = {name for name, _ in PREF_SET_B}
assert len(PREF_SET_B) == 11
print(f"Phase A: {len(PREF_SET)} preferences; Phase B: {len(PREF_SET_B)} named preferences")


In [ ]:
# Cell 14
def norm_R(v):
    return float(np.sqrt(max(v @ R @ v, 0.0)))

rows = []
for pname, p in PREF_SET:
    base = {"p_name": pname, **{f"p_{a}": float(p[i]) for i, a in enumerate(ATTRIBUTES)}}
    results = []
    for rho in RHO_GRID:
        results.append(("Avg", f"rho={rho}", avg(p, R, rho), "OK", {}))
    lam_cert, t_cert = cert(p, R, CERT_C, CERT_EPS)
    t_lp, collapsed = floor_lp_at_p(R, p)
    results.append((
        "Cert", "", lam_cert,
        "FLOOR_COLLAPSED" if collapsed else "FLOOR_NONTRIVIAL",
        {"t_star": t_cert, "t_star_lp": t_lp},
    ))
    for c in C_GRID:
        lam_mm, t_mm, status = maxmin_c(p, R, c)
        results.append(("MaxMin", f"c={c}", lam_mm, status, {"t_star": t_mm}))
    for alpha in ALPHA_GRID:
        for eps in EPS_GRID:
            lam_f, u_f, status = fair_alpha_eps(p, R, alpha, eps)
            results.append(("Fair", f"alpha={alpha},eps={eps}", lam_f, status, {"u_alpha": u_f}))

    for method, params, lam, status, extra in results:
        row = dict(base, method=method, params=params, status=status, **extra)
        if lam is None:
            row.update({f"lam_{a}": np.nan for a in ATTRIBUTES})
            row.update({"dist_l2": np.nan, "dist_R": np.nan, "proxy_pRlam": np.nan,
                        "min_delta": np.nan, "moved": False, "usable": False})
        else:
            lam = np.asarray(lam, dtype=np.float64)
            v = lam - p
            row.update({f"lam_{a}": float(lam[i]) for i, a in enumerate(ATTRIBUTES)})
            row.update({"dist_l2": float(np.linalg.norm(v)), "dist_R": norm_R(v),
                        "proxy_pRlam": float(p @ R @ lam),
                        "min_delta": float(np.min(improvements(p, R, lam))),
                        "moved": bool(np.linalg.norm(v) > 1e-8), "usable": True})
        rows.append(row)

lam_df_all = pd.DataFrame(rows)
lam_cols = [f"lam_{a}" for a in ATTRIBUTES]
p_cols = [f"p_{a}" for a in ATTRIBUTES]
lam_df_all.to_csv(LAMBDA_ALL_CSV, index=False)
print(f"All Phase-A rows including Cert audit: {len(lam_df_all)}")


In [ ]:
# Cell 15
# Cert exclusion is decided only from the eleven preferences that would be scored.
floor_by_preference, audit_rows = {}, []
for name, p in PREF_SET:
    t_lp, collapsed = floor_lp_at_p(R, p)
    cert_row = lam_df_all[(lam_df_all["p_name"] == name) & (lam_df_all["method"] == "Cert")].iloc[0]
    lam_cert = cert_row[lam_cols].to_numpy(float)
    distance = float(np.linalg.norm(lam_cert - p))
    equals_p = bool(distance <= FLOOR_TOL)
    assert float(cert_row["t_star"]) >= -FLOOR_TOL
    assert float(cert_row["t_star"]) <= max(float(t_lp), 0.0) + FLOOR_TOL
    assert float(cert_row["min_delta"]) >= -FLOOR_TOL
    if collapsed:
        assert equals_p, f"Floor LP collapsed but Cert moved for {name}: L2={distance}"
    floor_by_preference[name] = {"t_lp": float(t_lp), "collapsed": bool(collapsed)}
    audit_rows.append({
        "p_name": name, "phase_b": name in PHASE_B_NAMES,
        "floor_lp_collapsed": bool(collapsed), "lambda_equals_p": equals_p,
        "lambda_minus_p_l2": distance, "t_star_lp": float(t_lp),
        "t_star_cert": float(cert_row["t_star"]),
    })
cert_audit_df = pd.DataFrame(audit_rows)
cert_audit_df.to_csv(CERT_AUDIT_CSV, index=False)
phase_b_audit = cert_audit_df[cert_audit_df["phase_b"]]
CERT_EXCLUDED = should_exclude_cert_from_phase_b(
    cert_audit_df.to_dict(orient="records"), PHASE_B_NAMES, tolerance=FLOOR_TOL
)
CERT_POLICY = (
    "excluded_after_all Phase-B floors collapsed and lambda_cert equaled p"
    if CERT_EXCLUDED else
    "included because at least one Phase-B floor/candidate was non-trivial"
)
cert_audit_report = {
    "tolerance": FLOOR_TOL,
    "decision_scope": "all eleven named Phase-B preferences",
    "cert_excluded_from_reward_evaluation": CERT_EXCLUDED,
    "policy": CERT_POLICY,
    "phase_a_collapsed": int(cert_audit_df["floor_lp_collapsed"].sum()),
    "phase_a_total": int(len(cert_audit_df)),
    "phase_b_collapsed": int(phase_b_audit["floor_lp_collapsed"].sum()),
    "phase_b_lambda_equals_p": int(phase_b_audit["lambda_equals_p"].sum()),
    "phase_b_total": int(len(phase_b_audit)),
}
write_json(CERT_AUDIT_JSON, cert_audit_report)

lam_df = lam_df_all[lam_df_all["method"] != "Cert"].copy() if CERT_EXCLUDED else lam_df_all.copy()
lam_df.to_csv(LAMBDA_CSV, index=False)
assert ("Cert" not in set(lam_df["method"])) == CERT_EXCLUDED
print(json.dumps(cert_audit_report, indent=2))
print(f"Rows sent to method evaluation: {len(lam_df)}; Cert policy: {CERT_POLICY}")


In [ ]:
# Cell 16
# General feasibility checks apply after the conditional Cert removal.
usable = lam_df[lam_df["usable"]]
values = usable[lam_cols].to_numpy(float)
assert np.all(np.isfinite(values))
assert np.all(values >= -FLOOR_TOL)
assert np.allclose(values.sum(axis=1), 1.0, atol=FLOOR_TOL)
assert not (lam_df["status"] == "SOLVER_FAILED").any()
fair_rows = lam_df[(lam_df["method"] == "Fair") & lam_df["usable"]]
for _, row in fair_rows.iterrows():
    eps = float(row["params"].split("eps=")[1])
    assert float(row["min_delta"]) >= -eps - FLOOR_TOL

summary = (lam_df.assign(key=lam_df["method"] + lam_df["params"].map(lambda s: f"({s})" if s else ""))
           .groupby("key").agg(n=("moved", "size"), n_usable=("usable", "sum"),
                                n_moved=("moved", "sum"), mean_dist_l2=("dist_l2", "mean"))
           .sort_values("mean_dist_l2", ascending=False))
display(summary.round(5))
print(lam_df["status"].value_counts())


**Cell 17**

## 6. Fixed, disjoint evaluation prompts and unique merge points


In [ ]:
# Cell 18
from datasets import load_dataset
from scripts.diff_experts import select_unique_prompts
from src.eval_prompts import build_eval_prompt_file, ensure_nb06_prompt_files

validation_rows = load_dataset(DATASET_NAME, split="validation", revision=DATASET_REVISION)
nb11_diagnostic_prompts = select_unique_prompts(validation_rows, n_prompts=64, seed=991)
prompt_payload = json.dumps(nb11_diagnostic_prompts, ensure_ascii=False, separators=(",", ":"))
reconstructed_hash = hashlib.sha256(prompt_payload.encode("utf-8")).hexdigest()
assert reconstructed_hash == NB11_RECONSTRUCTED_DIAGNOSTIC_PROMPT_SHA256, (
    f"NB11 diagnostic reconstruction changed: {reconstructed_hash}"
)
nb11_historical_hash_match = reconstructed_hash == NB11_REPORTED_DIAGNOSTIC_PROMPT_SHA256
exclusion_rows = [{
    "prompt_id": f"nb11_diagnostic_{index:03d}",
    "category": "nb11_posthoc_differentiation_reconstructed",
    "prompt": prompt,
    "notes": f"Pinned selector/dataset; SHA256={reconstructed_hash}",
} for index, prompt in enumerate(nb11_diagnostic_prompts, start=1)]
if DIAGNOSTIC_EXCLUSION_PATH.exists():
    existing = [json.loads(line) for line in DIAGNOSTIC_EXCLUSION_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert [row["prompt"] for row in existing] == nb11_diagnostic_prompts
else:
    with DIAGNOSTIC_EXCLUSION_PATH.open("w", encoding="utf-8") as handle:
        for row in exclusion_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

ensure_nb06_prompt_files(
    PROJECT_ROOT, dataset_name=DATASET_NAME, dataset_revision=DATASET_REVISION,
    split="validation", seed=137, n_per_set=80,
)
PROMPT_SUMMARY = build_eval_prompt_file(
    REWARD_PROMPT_PATH, n=N_EVAL_PROMPTS, seed=EVAL_PROMPT_SEED,
    split="validation", dataset_name=DATASET_NAME, dataset_revision=DATASET_REVISION,
    prompt_id_prefix="nb13_1", project_root=PROJECT_ROOT,
    extra_exclude_paths=[DIAGNOSTIC_EXCLUSION_PATH], allow_missing_exclusions=False,
)
reward_prompts = [json.loads(line) for line in REWARD_PROMPT_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(reward_prompts) == N_EVAL_PROMPTS
assert not ({row["prompt"] for row in reward_prompts} & set(nb11_diagnostic_prompts))
assert PROMPT_SUMMARY["disjointness_verified"]
PROMPT_BINDING = {
    "dataset_name": DATASET_NAME, "dataset_revision": DATASET_REVISION,
    "split": "validation", "seed": EVAL_PROMPT_SEED, "n": N_EVAL_PROMPTS,
    "prompt_file_sha256": sha256_file(REWARD_PROMPT_PATH),
    "nb11_exclusion_file_sha256": sha256_file(DIAGNOSTIC_EXCLUSION_PATH),
    "nb11_reconstructed_prompt_list_sha256": reconstructed_hash,
    "nb11_reported_prompt_list_sha256": NB11_REPORTED_DIAGNOSTIC_PROMPT_SHA256,
    "nb11_historical_hash_match": nb11_historical_hash_match,
    "disjointness_verified": True,
}
print(json.dumps(PROMPT_BINDING, indent=2))


In [ ]:
# Cell 19
eval_points, origins = [], {}

def register_point(vec, origin):
    key = lambda_key(vec, decimals=LAMBDA_DEDUP_DECIMALS)
    if key not in origins:
        origins[key] = []
        eval_points.append(np.asarray(vec, dtype=np.float64))
    origins[key].append(origin)

lam_df_B = lam_df[lam_df["p_name"].isin(PHASE_B_NAMES)].copy()
assert ("Cert" not in set(lam_df_B["method"])) == CERT_EXCLUDED
for _, row in lam_df_B.iterrows():
    register_point(row[p_cols].to_numpy(float), f"baseline:{row['p_name']}")
    if bool(row["usable"]) and bool(row["moved"]):
        register_point(row[lam_cols].to_numpy(float), f"{row['method']}({row['params']}):{row['p_name']}")
EVAL_POINTS = np.asarray(eval_points, dtype=np.float64)
n_unique, n_prompts = len(EVAL_POINTS), len(reward_prompts)
print(f"Cert excluded:              {CERT_EXCLUDED}")
print(f"Phase-B method rows:        {len(lam_df_B)}")
print(f"Unique merge points:        {n_unique}")
print(f"Prompts per point:          {n_prompts}")
print(f"Generated answers total:    {n_unique * n_prompts}")
print(f"RM head scores total:       {3 * n_unique * n_prompts * m}")


**Cell 20**

## 7. Freeze the complete protocol before generation or reward contact


In [ ]:
# Cell 21
import platform
from importlib import metadata

def runtime_versions():
    names = ("torch", "transformers", "tokenizers", "peft", "accelerate",
             "datasets", "huggingface_hub", "bitsandbytes", "numpy", "pandas", "scipy")
    return {"python": platform.python_version(), **{name: metadata.version(name) for name in names}}

def protocol_notebook_sha256(path):
    text = Path(path).read_text(encoding="utf-8")
    text = re.sub(r"(PREREG_CONFIRM|RUN_REWARD_COLLECTION) = (True|False)",
                  r"\1 = <ACTIVATION_FLAG>", text)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

BINDING = {
    "schema_version": 1,
    "notebook": "NB13.1 HelpSteer2 DPO method comparison with three reward models",
    "execution_mode": "generate_once_score_three_rms",
    "claim_status": CLAIM_STATUS,
    "regime": REGIME,
    "adapter_bundle_sha256": ADAPTER_BUNDLE_SHA256,
    "mean_adapter_weights_sha256": {a: MEAN_MANIFESTS[a]["adapter_model_sha256"] for a in ATTRIBUTES},
    "mean_adapter_binding_sha256": {a: MEAN_MANIFESTS[a]["binding_sha256"] for a in ATTRIBUTES},
    "base_model": {"name": BASE_MODEL_NAME, "revision": BASE_REVISION},
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "training_split": "train", "evaluation_split": "validation"},
    "reward_models": {
        "armorm": {"name": ARMORM_MODEL, "revision": ARMORM_REVISION, "precision": "int8",
                   "role": "primary post-hoc evaluator"},
        "urm": {"name": URM_MODEL, "revision": URM_REVISION, "precision": "int8",
                "role": "sensitivity only; directly trained on HelpSteer2"},
        "steerlm": {"name": STEERLM_MODEL, "revision": STEERLM_REVISION,
                    "checkpoint_sha256": STEERLM_CHECKPOINT_SHA256,
                    "backend": "official NeMo-Aligner PyTriton server",
                    "role": "external robustness evaluator"},
    },
    "attribute_order": list(ATTRIBUTES),
    "geometry": {"primary_matrix": PRIMARY_MATRIX,
                 "R_cos_sha256": sha256_file(COSINE_MATRIX_CSV),
                 "R_gram_sha256": sha256_file(GRAM_MATRIX_CSV)},
    "prompts": PROMPT_BINDING,
    "generation": {"max_new_tokens": MAX_NEW_TOKENS,
                   "repetition_penalty": REPETITION_PENALTY,
                   "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
                   "decoding": "greedy; do_sample=False; num_beams=1",
                   "prompt_format": "TinyLlama tokenizer chat template",
                   "merge_dtype": "float32"},
    "grids": {"rho_grid": RHO_GRID, "cert_c": CERT_C, "cert_eps": CERT_EPS,
              "c_grid": C_GRID, "alpha_grid": ALPHA_GRID, "eps_grid": EPS_GRID},
    "cert_policy": {**cert_audit_report, "audit_sha256": sha256_file(CERT_AUDIT_CSV)},
    "n_unique_merge_points": int(n_unique),
    "source_sha256": {name: sha256_file(PROJECT_ROOT / "src" / name) for name in (
        "coefficient_portfolio.py", "cert_floor_policy.py", "merge.py", "proxy_validation.py", "metrics.py",
        "lambda_utils.py", "armorm_scorer.py", "armorm_objectives.py", "urm_scorer.py",
        "steerlm_scorer.py", "eval_prompts.py", "effective_lora_geometry.py")},
    "protocol_notebook_sha256": protocol_notebook_sha256(NOTEBOOK_PATH),
    "runtime_versions": runtime_versions(),
}
BINDING_SHA256 = hashlib.sha256(json.dumps(BINDING, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
lambda_hash = hashlib.sha256(
    lam_df[["p_name", "method", "params", "status"] + lam_cols].round(9).to_csv(index=False).encode()
).hexdigest()
prereg = {
    "schema_version": 1, "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "binding": BINDING, "binding_sha256": BINDING_SHA256,
    "lambda_table_sha256": lambda_hash,
    "methods_evaluated": sorted(lam_df["method"].unique().tolist()),
    "cert_excluded": CERT_EXCLUDED,
    "cert_exclusion_rule": "exclude iff all Phase-B floor LPs collapse and lambda_cert equals p within tolerance",
    "metric_primary": "within each RM: raw U_p = p^T r; never pooled across RMs",
    "reported_scale": "ArmoRM additionally mapped by 5r-0.5; URM and SteerLM retain native regression scale",
    "error_layer": "paired bootstrap over the same 80 prompts; 10000 draws",
    "multiplicity": "Holm both within each RM and globally across all RM/method/preference tests",
    "evaluator_roles": {k: v["role"] for k, v in BINDING["reward_models"].items()},
    "raw_answers_retained": True,
}
if PREREG_JSON.exists():
    frozen = json.loads(PREREG_JSON.read_text(encoding="utf-8"))
    print(f"Existing preregistration: {frozen['created_utc']}")
elif PREREG_CONFIRM:
    write_json(PREREG_JSON, prereg)
    frozen = prereg
    print(f"Preregistration frozen -> {PREREG_JSON}")
else:
    frozen = None
    print(json.dumps(prereg, indent=2)[:4000])

GATE_OPEN = bool(
    frozen
    and frozen.get("binding_sha256") == BINDING_SHA256
    and frozen.get("lambda_table_sha256") == lambda_hash
)
print("GATE OPEN" if GATE_OPEN else "GATE CLOSED")
if RUN_REWARD_COLLECTION and not GATE_OPEN:
    raise RuntimeError("Generation/scoring requested but the frozen NB13.1 binding does not match.")


**Cell 22**

## 8. Generate every frozen answer once and cache it on Drive


In [ ]:
# Cell 23
if RUN_REWARD_COLLECTION:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from src.merge import combine_effective_deltas, effective_deltas, resolve_base_module

    generation_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, revision=BASE_REVISION, use_fast=True)
    generation_tokenizer.pad_token = generation_tokenizer.eos_token
    generation_tokenizer.padding_side = "left"
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, revision=BASE_REVISION, torch_dtype=torch.float32, device_map="auto"
    ).eval()
    DELTAS = effective_deltas(dict(ADAPTER_PATHS))

    @contextmanager
    def merged_model(model, deltas_by_adapter, lam):
        merged = combine_effective_deltas(lam, deltas_by_adapter)
        originals = {}
        try:
            for module_name, delta_cpu in merged.items():
                module = resolve_base_module(model, module_name)
                originals[module_name] = module.weight.detach().clone()
                update = module.weight.detach().float() + delta_cpu.to(module.weight.device, torch.float32)
                with torch.no_grad():
                    module.weight.copy_(update.to(dtype=module.weight.dtype))
            yield model
        finally:
            for module_name, original in originals.items():
                with torch.no_grad():
                    resolve_base_module(model, module_name).weight.copy_(original)
            del originals, merged
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    def generate_answer(prompt):
        device = next(base_model.parameters()).device
        rendered = generation_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
        )
        encoded = generation_tokenizer(rendered, return_tensors="pt", add_special_tokens=False,
                                       truncation=True, max_length=512)
        input_ids = encoded["input_ids"].to(device)
        with torch.inference_mode():
            generated = base_model.generate(
                input_ids=input_ids, attention_mask=encoded["attention_mask"].to(device),
                max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                pad_token_id=generation_tokenizer.pad_token_id,
                eos_token_id=generation_tokenizer.eos_token_id,
            )
        return generation_tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True).strip()
    print("TinyLlama generation model and exact mean-adapter deltas are ready.")


In [ ]:
# Cell 24
def read_repair_jsonl(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return []
    valid = []
    raw_lines = path.read_text(encoding="utf-8").splitlines()
    for index, line in enumerate(raw_lines):
        if not line.strip():
            continue
        try:
            valid.append(json.loads(line))
        except json.JSONDecodeError:
            if index != len(raw_lines) - 1:
                raise
            print(f"[repair] Removing interrupted final JSONL line from {path.name}")
            path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in valid), encoding="utf-8")
    return valid

def prompt_digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

if RUN_REWARD_COLLECTION:
    answer_header = {
        "kind": "nb13_1_answer_cache", "binding_sha256": BINDING_SHA256,
        "num_points": int(n_unique), "num_prompts": int(n_prompts),
        "prompt_file_sha256": sha256_file(REWARD_PROMPT_PATH),
    }
    existing = read_repair_jsonl(ANSWER_CACHE)
    if not existing:
        ANSWER_CACHE.write_text(json.dumps(answer_header) + "\n", encoding="utf-8")
        existing = [answer_header]
    assert existing[0] == answer_header, "Answer cache belongs to a different binding. Use a new RUN_TAG."
    completed = {str(row["key"]): row for row in existing[1:] if "key" in row}
    prompt_ids = [row["prompt_id"] for row in reward_prompts]
    prompt_hashes = [prompt_digest(row["prompt"]) for row in reward_prompts]
    missing_initial = sum(coefficient_key(point) not in completed for point in EVAL_POINTS)
    started, new_done = time.perf_counter(), 0
    with ANSWER_CACHE.open("a", encoding="utf-8") as handle:
        for index, point in enumerate(EVAL_POINTS):
            key = coefficient_key(point)
            if key in completed:
                record = completed[key]
                assert record["prompt_ids"] == prompt_ids and record["prompt_sha256"] == prompt_hashes
                assert len(record["answers"]) == n_prompts
                continue
            point_started = time.perf_counter()
            with merged_model(base_model, DELTAS, point):
                answers = [generate_answer(row["prompt"]) for row in reward_prompts]
            record = {"key": key, "index": index, "lambda": point.tolist(),
                      "prompt_ids": prompt_ids, "prompt_sha256": prompt_hashes, "answers": answers}
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush(); os.fsync(handle.fileno())
            completed[key] = record
            new_done += 1
            elapsed = time.perf_counter() - started
            remaining = missing_initial - new_done
            eta = elapsed / max(new_done, 1) * remaining
            print(f"[generate] {index + 1}/{n_unique} point={time.perf_counter()-point_started:.1f}s "
                  f"remaining~{timedelta(seconds=int(eta))}")

    assert len(completed) == n_unique
    ANSWER_RECORDS = {coefficient_key(point): completed[coefficient_key(point)] for point in EVAL_POINTS}
    ANSWER_CACHE_SHA256 = sha256_file(ANSWER_CACHE)
    print(f"[OK] Frozen answers: {n_unique * n_prompts}; SHA256={ANSWER_CACHE_SHA256}")

    # No evaluator is loaded while generation is active. Release TinyLlama first.
    del base_model, generation_tokenizer, DELTAS
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Generation model released before reward scoring.")


**Cell 25**

## 9. Resumable evaluator collector

Each evaluator gets an independent cache and tensor. Its cache header is bound
to both the frozen experiment and the complete answer-cache hash.


In [ ]:
# Cell 26
RM_TENSORS, RM_DESCRIPTIONS, RM_CACHE_PATHS = {}, {}, {}

def collect_evaluator_tensor(evaluator, scorer, description):
    evaluator_binding = {
        "experiment_binding_sha256": BINDING_SHA256,
        "answer_cache_sha256": ANSWER_CACHE_SHA256,
        "evaluator": evaluator,
        "description": description,
        "attribute_order": list(ATTRIBUTES),
    }
    evaluator_binding_sha256 = hashlib.sha256(
        json.dumps(evaluator_binding, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    cache = PERSISTENT_DIR / f"reward_cache_{evaluator}.jsonl"
    header = {"kind": "nb13_1_reward_cache", "binding_sha256": evaluator_binding_sha256,
              "num_points": int(n_unique), "num_prompts": int(n_prompts), "num_attributes": m}
    existing = read_repair_jsonl(cache)
    if not existing:
        cache.write_text(json.dumps(header) + "\n", encoding="utf-8")
        existing = [header]
    assert existing[0] == header, f"{evaluator} cache belongs to a different binding."
    completed = {str(row["key"]): row["scores"] for row in existing[1:] if "key" in row}
    tensor = np.full((n_unique, n_prompts, m), np.nan, dtype=np.float64)
    missing_initial = sum(coefficient_key(point) not in completed for point in EVAL_POINTS)
    started, new_done = time.perf_counter(), 0
    with cache.open("a", encoding="utf-8") as handle:
        for index, point in enumerate(EVAL_POINTS):
            key = coefficient_key(point)
            if key in completed:
                scores = np.asarray(completed[key], dtype=np.float64)
            else:
                answer_record = ANSWER_RECORDS[key]
                if hasattr(scorer, "score_many"):
                    scores = np.asarray(scorer.score_many(
                        [row["prompt"] for row in reward_prompts],
                        answer_record["answers"], ATTRIBUTES,
                    ), dtype=np.float64)
                else:
                    scores = np.asarray([
                        scorer(row["prompt"], answer, ATTRIBUTES)
                        for row, answer in zip(reward_prompts, answer_record["answers"])
                    ], dtype=np.float64)
                if scores.shape != (n_prompts, m) or not np.all(np.isfinite(scores)):
                    raise RuntimeError(f"{evaluator} returned invalid score array {scores.shape}")
                handle.write(json.dumps({"key": key, "index": index, "lambda": point.tolist(),
                                         "scores": scores.tolist()}) + "\n")
                handle.flush(); os.fsync(handle.fileno())
                completed[key] = scores.tolist()
                new_done += 1
                elapsed = time.perf_counter() - started
                eta = elapsed / max(new_done, 1) * (missing_initial - new_done)
                print(f"[{evaluator}] {index + 1}/{n_unique} mean={np.round(scores.mean(0), 4)} "
                      f"remaining~{timedelta(seconds=int(eta))}")
            if scores.shape != (n_prompts, m):
                raise RuntimeError(f"Cached {evaluator} scores have shape {scores.shape}")
            tensor[index] = scores
    if np.isnan(tensor).any():
        raise RuntimeError(f"{evaluator} tensor remains incomplete")
    npy_path = RESULTS_DIR / f"reward_tensor_{evaluator}.npy"
    np.save(npy_path, tensor)
    RM_TENSORS[evaluator] = tensor
    RM_DESCRIPTIONS[evaluator] = description
    RM_CACHE_PATHS[evaluator] = cache
    print(f"[OK] {evaluator}: tensor={tensor.shape}, cache={cache}")
    return tensor

def release_evaluator(scorer):
    if hasattr(scorer, "unload"):
        scorer.unload()
    elif hasattr(scorer, "_model"):
        scorer._model = None
        if hasattr(scorer, "_tokenizer"):
            scorer._tokenizer = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


**Cell 27**

## 10. Score with ArmoRM, URM, and SteerLM-RM


In [ ]:
# Cell 28 — ArmoRM (primary)
if RUN_REWARD_COLLECTION:
    from src.armorm_scorer import make_score_prompt_answer
    armorm_score, armorm = make_score_prompt_answer(
        ARMORM_MODEL, revision=ARMORM_REVISION, dtype="bfloat16", load_in_8bit=True
    )
    armorm.assert_golden_sample()
    armorm_description = {**armorm.describe(),
        "role": "primary post-hoc evaluator",
        "native_to_0_4": "5 * raw - 0.5",
        "training_overlap": "HelpSteer objective heads; not trained on NB11.1 generations"}
    collect_evaluator_tensor("armorm", armorm_score, armorm_description)
    release_evaluator(armorm)
    del armorm_score, armorm


In [ ]:
# Cell 29 — URM (sensitivity; direct HelpSteer2 training overlap)
if RUN_REWARD_COLLECTION:
    from src.urm_scorer import URMScorer
    urm = URMScorer(URM_MODEL, revision=URM_REVISION, load_in_8bit=True, dtype="float16")
    urm.assert_golden_sample()
    urm_description = {**urm.describe(),
        "role": "sensitivity only",
        "native_to_0_4": "identity; raw means are not clipped",
        "training_overlap": "URM attribute head was trained directly on nvidia/HelpSteer2"}
    collect_evaluator_tensor("urm", urm.score, urm_description)
    release_evaluator(urm)
    del urm


**Cell 30 — Required SteerLM server preparation**

The published SteerLM-RM is not a Hugging Face Transformers checkpoint. Start
the frozen `.nemo` checkpoint in NVIDIA's NeMo-Aligner container on a suitable
multi-GPU machine, following the model card:

```bash
python /opt/NeMo-Aligner/examples/nlp/gpt/serve_reward_model.py \
  rm_model_file=Llama2-13B-SteerLM-RM.nemo \
  trainer.num_nodes=1 trainer.devices=8 \
  ++model.tensor_model_parallel_size=4 \
  ++model.pipeline_model_parallel_size=1 \
  inference.micro_batch_size=2 inference.port=1424
```

Expose the server to this runtime (prefer an authenticated tunnel) and set two
Colab secrets or environment variables:

- `STEERLM_HOST`: server host, without scheme or port;
- `STEERLM_OPERATOR_ATTESTATION`:
  `nvidia/Llama2-13B-SteerLM-RM@a8fd6a92a4967f934612f519a2ef078bb87ec431:sha256=19851451def71de48d044bbebf93481308f1a778153e96e8b93f490047082b27`

The official endpoint does not report its checkpoint identity. The explicit
attestation prevents an accidental silent substitution, but remains an
operator assertion. NB13.1 preserves raw nine-head regression output and then
selects Helpfulness, Correctness, Coherence, Complexity, and Verbosity.


In [ ]:
# Cell 31 — SteerLM-RM (external robustness)
if RUN_REWARD_COLLECTION:
    from src.steerlm_scorer import SteerLMRemoteScorer
    try:
        from google.colab import userdata
        secret_host = userdata.get("STEERLM_HOST")
        secret_attestation = userdata.get("STEERLM_OPERATOR_ATTESTATION")
    except Exception:
        secret_host = secret_attestation = None
    STEERLM_HOST = os.environ.get("STEERLM_HOST") or secret_host
    STEERLM_OPERATOR_ATTESTATION = (
        os.environ.get("STEERLM_OPERATOR_ATTESTATION") or secret_attestation
    )
    if not STEERLM_HOST or not STEERLM_OPERATOR_ATTESTATION:
        raise RuntimeError(
            "SteerLM is the required third evaluator. Start the official NeMo server "
            "and set STEERLM_HOST plus STEERLM_OPERATOR_ATTESTATION as described above. "
            "ArmoRM/URM and all generated answers are already cached; rerunning resumes."
        )
    steerlm = SteerLMRemoteScorer(
        host=STEERLM_HOST, port=1424, server_model_name="reward_model",
        operator_attestation=STEERLM_OPERATOR_ATTESTATION,
    )
    steerlm.health_check()
    steerlm_description = {**steerlm.describe(),
        "role": "external robustness evaluator",
        "native_to_0_4": "identity; raw regression outputs are not rounded or clipped",
        "training_overlap": "trained on original HelpSteer and OASST1, not HelpSteer2"}
    collect_evaluator_tensor("steerlm", steerlm, steerlm_description)
    del steerlm

assert set(RM_TENSORS) == {"armorm", "urm", "steerlm"}, (
    f"All three evaluators are required; completed={sorted(RM_TENSORS)}"
)
assert all(tensor.shape == (n_unique, n_prompts, m) for tensor in RM_TENSORS.values())


**Cell 32**

## 11. Per-RM method tables and paired statistics

No raw reward is averaged across evaluators. ArmoRM is additionally converted
to its documented HelpSteer 0–4 scale; URM and SteerLM retain their native
regression scales without clipping.


In [ ]:
# Cell 33
point_index = {
    lambda_key(point, decimals=LAMBDA_DEDUP_DECIMALS): index
    for index, point in enumerate(EVAL_POINTS)
}

def point_i(vec):
    key = lambda_key(np.asarray(vec, dtype=float), decimals=LAMBDA_DEDUP_DECIMALS)
    if key not in point_index:
        raise KeyError(f"Missing evaluated merge point: {key}")
    return point_index[key]

def to_reported_0_4(evaluator, value):
    return 5.0 * float(value) - 0.5 if evaluator == "armorm" else float(value)

final_rows = []
for evaluator, tensor in RM_TENSORS.items():
    means = tensor.mean(axis=1)
    for _, row in lam_df_B.iterrows():
        p = row[p_cols].to_numpy(float)
        usable = bool(row["usable"])
        lam = row[lam_cols].to_numpy(float) if usable else None
        u_base = float(means[point_i(p)] @ p)
        u_lam = float(means[point_i(lam)] @ p) if usable else np.nan
        u_base_04 = to_reported_0_4(evaluator, u_base)
        u_lam_04 = to_reported_0_4(evaluator, u_lam) if usable else np.nan
        pct = 100.0 * (u_lam_04 - u_base_04) / abs(u_base_04) if abs(u_base_04) > 1e-12 else np.nan
        final_rows.append({
            "evaluator": evaluator, "evaluator_role": RM_DESCRIPTIONS[evaluator]["role"],
            "p_name": row["p_name"], "p": np.round(p, 6).tolist(),
            "method": row["method"], "params": row["params"], "status": row["status"],
            "lambda": np.round(lam, 6).tolist() if usable else None,
            "dist_l2": row["dist_l2"], "dist_R": row["dist_R"],
            "proxy_delta_p": float(p @ R @ (lam - p)) if usable else np.nan,
            "proxy_min_delta": row["min_delta"],
            "U_p(p)": u_base, "U_p(lambda)": u_lam, "Delta U_p": u_lam - u_base,
            "U_p(p)_0_4": u_base_04, "U_p(lambda)_0_4": u_lam_04,
            "Delta U_p_percent": pct,
            "note": ("same merge point as baseline" if usable and np.linalg.norm(lam-p) <= 1e-8
                     else (row["status"] if not usable else "")),
        })
final_df = pd.DataFrame(final_rows)
assert ("Cert" not in set(final_df["method"])) == CERT_EXCLUDED
final_df.to_csv(FINAL_CSV, index=False)
print(f"{len(final_df)} rows -> {FINAL_CSV}")
display(final_df.head(30).round(5))


In [ ]:
# Cell 34
from src.lambda_utils import holm_adjust
from src.metrics import paired_bootstrap_ci, selection_regret

stats_rows = []
for evaluator, tensor in RM_TENSORS.items():
    means = tensor.mean(axis=1)
    for _, row in lam_df_B.iterrows():
        if not (bool(row["usable"]) and bool(row["moved"])):
            continue
        p = row[p_cols].to_numpy(float)
        lam = row[lam_cols].to_numpy(float)
        i_lam, i_base = point_i(lam), point_i(p)
        boot = paired_bootstrap_ci(tensor[i_lam], tensor[i_base], p)
        u_base = float(means[i_base] @ p)
        u_lam = float(means[i_lam] @ p)
        stats_rows.append({
            "evaluator": evaluator, "evaluator_role": RM_DESCRIPTIONS[evaluator]["role"],
            "p_name": row["p_name"], "method": row["method"], "params": row["params"],
            "U_p_baseline": u_base, "U_p_lambda": u_lam,
            **{key: boot[key] for key in ("delta_u_p", "ci_low", "ci_high", "excludes_zero", "p_value")},
            "selection_regret": selection_regret(u_lam, (means @ p).tolist()),
            "baseline_regret": selection_regret(u_base, (means @ p).tolist()),
        })
stats_df = pd.DataFrame(stats_rows)
assert "Cert" not in set(stats_df["method"]) if CERT_EXCLUDED else True
stats_df["p_holm_within_rm"] = np.nan
for evaluator, indices in stats_df.groupby("evaluator").groups.items():
    stats_df.loc[indices, "p_holm_within_rm"] = holm_adjust(stats_df.loc[indices, "p_value"].to_numpy(float))
stats_df["p_holm_global"] = holm_adjust(stats_df["p_value"].to_numpy(float))
stats_df["holm_within_rm_improves"] = (stats_df["delta_u_p"] > 0) & (stats_df["p_holm_within_rm"] < 0.05)
stats_df["holm_global_improves"] = (stats_df["delta_u_p"] > 0) & (stats_df["p_holm_global"] < 0.05)
stats_df["holm_global_harms"] = (stats_df["delta_u_p"] < 0) & (stats_df["p_holm_global"] < 0.05)
stats_df.to_csv(STATS_CSV, index=False)
display(stats_df.groupby("evaluator").agg(
    n=("delta_u_p", "size"), mean_delta=("delta_u_p", "mean"),
    n_positive=("delta_u_p", lambda x: int((x > 0).sum())),
    n_global_improves=("holm_global_improves", "sum"),
    n_global_harms=("holm_global_harms", "sum"),
).round(5))
print(f"Global Holm family: {len(stats_df)} moving comparisons across all three RMs.")


**Cell 35**

## 12. Normalization, proxy validity, and cross-RM agreement


In [ ]:
# Cell 36
robust_rows, proxy_rows, proxy_reports = [], [], {}
for evaluator, tensor in RM_TENSORS.items():
    matrix = tensor.mean(axis=1)
    for pname, p in PREF_SET_B:
        agreement = normalization_agreement(matrix, p)
        robust_rows.append({
            "evaluator": evaluator, "p_name": pname,
            "argmax_agrees": agreement["argmax_agrees"],
            **{f"argmax_{k}": v for k, v in agreement["argmax_index"].items()},
            **{f"rho_{k}": v for k, v in agreement["spearman"].items()},
        })
    proxy_reports[evaluator] = run_spearman_analysis(
        EVAL_POINTS, matrix, R_cos, R_gram, {name: p for name, p in PREF_SET_B}
    )
    for _, row in lam_df_B.iterrows():
        if not (bool(row["usable"]) and bool(row["moved"])):
            continue
        p, lam = row[p_cols].to_numpy(float), row[lam_cols].to_numpy(float)
        proxy_rows.append({
            "evaluator": evaluator, "p_name": row["p_name"],
            "method": row["method"], "params": row["params"],
            "predicted_delta_u_p": float(p @ R @ (lam-p)),
            "predicted_min_axis_delta": float(np.min(R @ (lam-p))),
            "observed_delta_u_p": float((matrix[point_i(lam)] - matrix[point_i(p)]) @ p),
        })
robust_df, proxy_df = pd.DataFrame(robust_rows), pd.DataFrame(proxy_rows)
robust_df.to_csv(ROBUST_CSV, index=False)
proxy_df.to_csv(PROXY_CSV, index=False)
for evaluator, group in proxy_df.groupby("evaluator"):
    proxy_reports[evaluator]["method_delta_spearman"] = safe_spearman(
        group["predicted_delta_u_p"].to_numpy(float), group["observed_delta_u_p"].to_numpy(float)
    )
    proxy_reports[evaluator]["role"] = "descriptive only; never used to choose a grid point"
write_json(PROXY_JSON, proxy_reports)
display(robust_df.groupby("evaluator")["argmax_agrees"].agg(["sum", "count"]))


In [ ]:
# Cell 37
# Agreement uses signs and ranks only; raw magnitudes are never pooled across RMs.
comparison_key = ["p_name", "method", "params"]
pivot = stats_df.pivot(index=comparison_key, columns="evaluator", values="delta_u_p").reset_index()
required = [name for name in ("armorm", "urm", "steerlm") if name in pivot]
pivot["all_three_signs_agree"] = (
    np.ptp(np.sign(pivot[required].to_numpy(float)), axis=1) == 0
)
agreement_rows = []
for i, left in enumerate(required):
    for right in required[i + 1:]:
        agreement_rows.append({
            "left": left, "right": right, "n": len(pivot),
            "spearman_delta": safe_spearman(pivot[left].to_numpy(float), pivot[right].to_numpy(float)),
            "same_sign_fraction": float((np.sign(pivot[left]) == np.sign(pivot[right])).mean()),
        })
agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(RM_AGREEMENT_CSV, index=False)
display(agreement_df.round(5))
print(f"All-three sign agreement: {int(pivot['all_three_signs_agree'].sum())}/{len(pivot)} comparisons")


**Cell 38**

## 13. Final report, ZIP export, and download


In [ ]:
# Cell 39
report = {
    "schema_version": 1,
    "notebook": "NB13.1 HelpSteer2 DPO method comparison with three reward models",
    "run_tag": RUN_TAG, "created_utc": datetime.now(timezone.utc).isoformat(),
    "claim_status": CLAIM_STATUS, "regime": REGIME,
    "binding_sha256": BINDING_SHA256,
    "adapter_bundle_sha256": ADAPTER_BUNDLE_SHA256,
    "cert_excluded": CERT_EXCLUDED, "cert_policy": CERT_POLICY,
    "cert_audit": cert_audit_report,
    "n_preferences_phase_a": len(PREF_SET), "n_preferences_phase_b": len(PREF_SET_B),
    "n_phase_b_method_rows_per_rm": len(lam_df_B),
    "n_unique_merge_points": n_unique, "n_prompts": n_prompts,
    "n_frozen_answers": n_unique * n_prompts,
    "answer_cache_sha256": ANSWER_CACHE_SHA256,
    "reward_models": RM_DESCRIPTIONS,
    "reward_tensor_shapes": {name: list(tensor.shape) for name, tensor in RM_TENSORS.items()},
    "raw_scores_pooled_across_reward_models": False,
    "global_holm_family_size": len(stats_df),
    "files": {},
}
write_json(REPORT_JSON, report)

export_paths = [
    LAMBDA_ALL_CSV, LAMBDA_CSV, CERT_AUDIT_CSV, CERT_AUDIT_JSON,
    PREREG_JSON, ANSWER_CACHE, REWARD_PROMPT_PATH, DIAGNOSTIC_EXCLUSION_PATH,
    COSINE_MATRIX_CSV, GRAM_MATRIX_CSV, D_NORMS_CSV, GEOMETRY_JSON,
    FINAL_CSV, STATS_CSV, ROBUST_CSV, PROXY_CSV, PROXY_JSON,
    RM_AGREEMENT_CSV, REPORT_JSON,
    *RM_CACHE_PATHS.values(),
    *[RESULTS_DIR / f"reward_tensor_{name}.npy" for name in sorted(RM_TENSORS)],
]
report["files"] = {
    Path(path).name: sha256_file(path)
    for path in export_paths
    if Path(path).is_file() and Path(path) != REPORT_JSON
}
write_json(REPORT_JSON, report)

zip_path = RESULTS_DIR / f"{RUN_TAG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    used_names = set()
    for path in export_paths:
        path = Path(path)
        if not path.is_file():
            continue
        name = path.name
        if name in used_names:
            name = f"{path.parent.name}_{name}"
        used_names.add(name)
        archive.write(path, name)
zip_sha = sha256_file(zip_path)
shutil.copy2(zip_path, PERSISTENT_DIR / zip_path.name)
print(f"ZIP: {zip_path}\nSHA256: {zip_sha}")
print(f"Cert evaluated by RMs: {not CERT_EXCLUDED}")
print(f"Evaluators complete: {sorted(RM_TENSORS)}")

from google.colab import files
files.download(str(zip_path))


In [ ]:
# Cell 40
!git status --short
